# Model Inference Through the API

This notebook sends a sample patient record to the running FastAPI service. Start the
service first in a terminal from the repository root, for example:

```powershell
docker build -t heart-disease-prediction:latest .
docker run --rm -p 8000:8000 heart-disease-prediction:latest
```

Then run the cells below. The notebook calls the API only; it does not load model files
from the notebook process.

In [ ]:
import json
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

API_URL = "http://127.0.0.1:8000"

def get_json(path):
    with urlopen(f"{API_URL}{path}", timeout=10) as response:
        return json.loads(response.read().decode("utf-8"))

health = get_json("/health")
print("API health:", health)

## Submit a prediction

The fields and value ranges match the API input schema. The service returns the predicted
class, confidence, and risk level.

In [ ]:
sample_patient = {
    "age": 52,
    "sex": 1,
    "cp": 0,
    "trestbps": 125,
    "chol": 212,
    "fbs": 0,
    "restecg": 1,
    "thalach": 168,
    "exang": 0,
    "oldpeak": 1.0,
    "slope": 2,
    "ca": 2,
    "thal": 3,
}
request = Request(
    f"{API_URL}/predict",
    data=json.dumps(sample_patient).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
try:
    with urlopen(request, timeout=30) as response:
        prediction = json.loads(response.read().decode("utf-8"))
    print(json.dumps(prediction, indent=2))
except HTTPError as error:
    print(f"API returned HTTP {error.code}: {error.read().decode('utf-8')}")
    raise
except URLError as error:
    raise RuntimeError(
        "Could not reach the API. Start the Docker container or FastAPI service, "
        "then rerun this cell."
    ) from error

## Check monitoring metrics

The metrics endpoint exposes Prometheus counters and latency measurements for monitoring.

In [ ]:
with urlopen(f"{API_URL}/metrics", timeout=10) as response:
    metrics_text = response.read().decode("utf-8")
print("\n".join(
    line for line in metrics_text.splitlines()
    if line.startswith(("predictions_total", "prediction_latency_seconds"))
) or "No prediction metrics found yet.")

## Interpreting the response

`prediction` is `0` for no disease and `1` for disease according to this trained model.
`confidence` is the model's probability for the predicted class. `risk_level` is the
service's output category. This is a coursework demonstration and is not a clinical
diagnosis.